# Welcome!
This notebook demonstrates how to develop a conversational system that uses a deep knowledge base about hotels, combining structured instance-level data and an ontological model. The knowledge graph (KG) and ontology enable reasoning to enhance dialogue response generation. The task involves integrating a GraphRAG-like approach to query the knowledge base and generate accurate, context-aware responses.

Specifically, the notebook has the following steps:

1. **Setup**: Loading the knowledge graph, dialogues, and required libraries (e.g., OWLAPY).
2. **Analyzing the knowledge graph**: Exploring its structure and entities using OWLAPY queries.
3. **Extending the ontology**: Adding TBox information for expressive reasoning.
4. **Creating dialogues**: Create dialogues based on the examples. Write 5 simple dialogues and 5 more detailed ones to showcase different types of interactions.
5. **Combining ontology and KG data**: Deploying an OWL reasoner to perform class-expression queries.
6. **Query generation with LLMs**: Using an LLM (e.g., Llama3.2) to generate or assist in creating queries against the KG.
7. **Generating responses**: Summarizing retrieved data into dialogue responses using a KG-augmented RAG approach.
8. **Evaluation**: Assessing the system's performance using metrics like intersection-over-union scores.

## Assignment
The goal of this assignment is to develop a logic-enhanced conversational system that retrieves and reasons over domain knowledge to assist in dialogue response generation. You will focus on both the technical aspects of KG+ontology reasoning and the integration with LLMs for robust responses.

### Assignment Steps
1. **Analyze the provided knowledge graph and dialogues**:
   - Explore the KG's entities, properties, and relevance to the dialogues.
   - Identify opportunities where ontology reasoning enhances dialogue responses.
2. **Extend the ontology**:
   - Add expressive TBox information to support meaningful inferences.
3. **Deploy the reasoning environment**:
   - Use OWLAPY to combine the KG (as ABox) with the ontology for reasoning-based queries.
4. **Generate class-expression queries**:
   - Use instruction-based, few-shot prompting with Llama3.2 to produce or assist in creating the queries.
5. **Summarize results into dialogue responses**:
   - Apply KG-augmented RAG to generate user-facing answers based on reasoning results.
6. **Evaluate the system**:
   - Use appropriate metrics, including intersection-over-union scores for set-based answers.

## Report
Write a **5-page report** in LNCS format that includes:

1. **Introduction**: Background on conversational systems with LLMs and the role of reasoning over domain knowledge.
2. **Methodology**: A detailed description of your approach, including diagrams and examples.
3. **Results**: Evaluation findings from the implemented steps.
4. **Discussion**: Strengths and weaknesses of your approach, lessons learned, and potential improvements.

Make sure to use the following template: [Springer Lecture Notes in Computer Science](https://www.overleaf.com/latex/templates/springer-lecture-notes-in-computer-science/kzwwpvhwnvfj)


## Grading
Your work will be evaluated based on:

1. **Code Implementation (30%)**: Quality and functionality of the logic-enhanced conversational system.
2. **Report (70%)**: Depth of analysis and clarity in presenting methods, results, and lessons learned.

## Kaggle Environment Notes
To ensure smooth execution:
- Load the required data into `/kaggle/input/`.
- Use `/kaggle/working/` for saving temporary files.
- Turn on GPUs and internet connectivity when necessary, and follow best practices for resource management.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input director

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Install packages

In [ ]:
!pip install jpype1==1.5.2
!pip install owlapy==1.5.1
!pip install ollama

# Import libraries


In [31]:
!pip install owlapy==1.5.1


   ---------------------------------------- 0.0/17.2 MB ? eta -:--:--
   ----- ---------------------------------- 2.4/17.2 MB 12.3 MB/s eta 0:00:02
   ----------- ---------------------------- 5.0/17.2 MB 11.6 MB/s eta 0:00:02
   ----------------- ---------------------- 7.6/17.2 MB 11.7 MB/s eta 0:00:01
   ---------------------- ----------------- 9.7/17.2 MB 11.2 MB/s eta 0:00:01
   ----------------------------- ---------- 12.6/17.2 MB 11.3 MB/s eta 0:00:01
   ----------------------------------- ---- 15.2/17.2 MB 11.4 MB/s eta 0:00:01
   ---------------------------------------  17.0/17.2 MB 11.4 MB/s eta 0:00:01
   ---------------------------------------- 17.2/17.2 MB 10.4 MB/s  0:00:01
   ---------------------------------------- 0.0/14.7 MB ? eta -:--:--
   ------ --------------------------------- 2.4/14.7 MB 12.2 MB/s eta 0:00:02
   ------------- -------------------------- 5.0/14.7 MB 11.6 MB/s eta 0:00:01
   -------------------- ------------------- 7.6/14.7 MB 11.7 MB/s eta 0:00:01
 

In [32]:
from owlapy import manchester_to_owl_expression, dl_to_owl_expression
from owlapy.iri import IRI
from owlapy.owl_ontology import Ontology
from owlapy.owl_reasoner import SyncReasoner, StructuralReasoner

# 1. Analyze the provided knowledge graph (data.ttl).

In [ ]:
os.path.exists()

False

In [55]:
onto = Ontology(local_file)

AssertionError: Input must not contain whitespace. Currently:C:/Users/hippo/Documents/University/VU - Bachelor Artificial Intelligence - 2026/PSAI/Project-Situated-AI-2026/data.ttl.

In [54]:
# Use forward slashes manually in a raw string
local_file = r"C:/Users/hippo/Documents/University/VU - Bachelor Artificial Intelligence - 2026/PSAI/Project-Situated-AI-2026/data.ttl"

# Verify existence before passing it to the IRI class
import os
if os.path.exists(local_file):
    print("File found by OS!")
    onto = Ontology(IRI.create(iri=local_file, is_file_path=True), load=True)
else:
    print("OS still cannot see the file. Check for typos or hidden extensions (e.g., data.ttl.txt)")

File found by OS!


OwlReadyOntologyParsingError: NTriples parsing error (or unrecognized file format) in C:/Users/hippo/Documents/University/VU - Bachelor Artificial Intelligence - 2026/PSAI/Project-Situated-AI-2026/data.ttl, line 5.

# 2. Create a small ontology that can support expressive inference about hotels and analyse the dialogues (examples.txt).

# Create your own dialogues

Once you have created your ontology, use it as the foundation for designing dialogues. Study the examples in examples.txt to understand their structure and content. Then, create 10 dialogues of your own, ensuring a range of difficulty levels: 5 simple ones and 5 more challenging ones. These dialogues should illustrate how your ontology can support reasoning and should include references to the types of information modeled in your ontology.

In [ ]:
# Create 10 dialoges based on the description
dialogue1: str = ""
dialogue2: str = ""
dialogue3: str = ""
dialogue4: str = ""
dialogue5: str = ""
dialogue6: str = ""
dialogue7: str = ""
dialogue8: str = ""
dialogue9: str = ""
dialogue10: str = ""
dialogues: list = [dialogue1, dialogue2, dialogue3, dialogue4, dialogue5, dialogue6, dialogue7, dialogue8, dialogue9, dialogue10]

# 3. Deploy a reasoning environment

In [ ]:
ontology_path: str = "..." # your path (Kaggle, Colab or local)

In [ ]:
# TODO: load your ontology and create reasoner

# 4. Instruct the LLM to produce the query or components of the query (e.g., keywords) against the KG

In [ ]:
#Download ollama
# For Kaggle or Linux: download with this command, for Windows & Mac locally, download executable from website
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
process = subprocess.Popen("ollama serve", shell=True) #runs on a different thread

#Download Python library
!pip install ollama

In [ ]:
# Import ollama & pull LLM
import ollama
!ollama pull llama3.2
model: str = "llama3.2"

In [ ]:
#Step 1: Write the instruction for the LLM - remember the overarching topic (assistance with hotels), as well as the fact that
# this step is meant to merely extract queries from the user input.

# Instruct LLM
instruction: str = "..."

In [ ]:
#Step 2: Write a function that takes the model, instruction and one user question as input, runs the LLM and outputs its response
def question_to_query(instruction: str, question: str, model="llama3.2") -> str:
    '''
    This function is meant to use the instruction defined above to run the LLM in order to convert one user input
    question into a query for the ontology reasoner.
    Parameters: instruction (string), question (string), model version (string)
    Returns: LLM response (string)
    '''
    # TODO

In [ ]:
#Step 3: Run the LLM for each example defined above

# Helper function
def find_between(s: str, start: str, end: str) -> str:
    return s.split(start)[1].split(end)[0]

for dialogue in dialogues:
    print("User question:", dialogue)
    print()
    result: str = question_to_query(instruction, dialogue, model)
    # Possibly only extract the relevant parts
    print("Extracted query:", result)
    queries.append(result)
    print()

# 5. Use an LLM to summarize some result into a natural language response to the user.

In [ ]:
#Step 1: Extract knowledge from query with the reasoner and return as list
def reason(query: str) -> list:
    '''
    This function should convert a query into an OWL expression and use the reasoner
    to return the answers.
    '''
    # TODO

In [ ]:
#Step 2: Instruct & run the LLM for the new task: transform the extracted knowledge into a natural language response based
# on the original question
def knowledge_to_response(question: str, knowledge: str, model="llama3.2"):
    '''
    This function is meant to write an instruction based on an item of extracted knowledge and the original user
    question, and run the LLM to summarize a response.
    '''
    # TODO

In [ ]:
#Step 3: Combine everything: generate queries from the dialogues, extract knowledge from queries with the reasoner and
# generate summary responses

# 6. Evaluate your LLM

In [ ]:
# TODO: your code to implement and demonstrate evaluation metrics
# Suggestions: comparison of generated queries with the queries manually created in examples.txt, Intersection Over Union,
# but you can be creative here